# Multicoordinate Wave Project

The multicoordinate wave example generates one BHaH project with several coordinate systems. It reuses the wave-equation right-hand side, reference metrics, precompute support, curvilinear boundary conditions, Method-of-Lines timestepping, and diagnostics in one project.

[Index](../index.ipynb) | Previous: [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb) | Next: [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)

## Why This Matters

The multicoordinate wave project combines the wave equation, reference metrics, fisheye maps, and boundary data into one generated workflow.

## What You Will See

- Generator configuration constants.
- A generated multicoordinate file inventory.
- A `make` build and short executable run.
- Representative 0D diagnostic samples.

## Table of Contents

1. [Runtime context and generator discovery](#Runtime-context-and-generator-discovery)
2. [Generator constants](#Generator-constants)
3. [Project generation](#Project-generation)
4. [Generated file inventory](#Generated-file-inventory)
5. [Build and run](#Build-and-run)
6. [Related notebooks](#Related-notebooks)

## Runtime Context and Generator Discovery

Discovery uses `importlib.util.find_spec` so the packaged generator is not imported at notebook startup. Importing the generator module would execute generation-time code.

In [1]:
import ast
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy


print("nrpy import succeeded")

generator_name = "nrpy.examples.wave_equation_multicoordinates"
generator_spec = importlib.util.find_spec(generator_name)
print("Generator discovered:", generator_spec is not None)

nrpy import succeeded
Generator discovered: True


## Generator Constants

The generator source records the project name, coordinate systems, grid sizes, finite-difference order, radiation boundary order, Method-of-Lines method, and diagnostics choices. Reading those literal assignments is safe because it does not import or run the generator.

In [2]:
tracked_names = {
    "project_name",
    "set_of_CoordSystems",
    "Nxx_dict",
    "MoL_method",
    "fd_order",
    "radiation_BC_fd_order",
    "enable_KreissOliger_dissipation",
    "boundary_conditions_desc",
}

generator_config = {}
if generator_spec is None or generator_spec.origin is None:
    raise RuntimeError(f"Generator not found: {generator_name}")
else:
    source = Path(generator_spec.origin).read_text(encoding="utf-8")
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in tracked_names:
                    try:
                        generator_config[target.id] = ast.literal_eval(node.value)
                    except (ValueError, SyntaxError):
                        pass

for name in sorted(generator_config):
    value = generator_config[name]
    if isinstance(value, set):
        value = sorted(value)
    print(name + ":", value)

MoL_method: RK4
Nxx_dict: {'Spherical': [64, 2, 2], 'SinhSpherical': [64, 2, 2], 'Cartesian': [64, 64, 64], 'SinhCartesian': [64, 64, 64]}
boundary_conditions_desc: outgoing radiation
enable_KreissOliger_dissipation: False
fd_order: 4
project_name: wave_equation_multicoordinates
radiation_BC_fd_order: 2
set_of_CoordSystems: ['Cartesian', 'SinhCartesian', 'SinhSpherical', 'Spherical']


## Project Generation

Generation runs inside a disposable workspace, not from the repository root or an active project directory. The default generator mode is used so the generated project includes the support headers required by the build.

In [3]:
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy-wave-multicoord-")
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_multicoordinates"

print("Temporary workspace: temporary directory")
print("Project directory exists before generation:", project_path.exists())

generation_completed = False
if generator_spec is None:
    raise RuntimeError(f"Generator not found: {generator_name}")
else:
    generate = subprocess.run(
        [sys.executable, "-m", generator_name],
        cwd=workspace_path,
        check=False,
        capture_output=True,
        text=True,
        timeout=600,
    )
    generation_completed = generate.returncode == 0
    print("Generator return code:", generate.returncode)
    if generate.stdout.strip():
        print("\n".join(generate.stdout.splitlines()[:16]))
    if generate.returncode != 0 and generate.stderr.strip():
        print("Generator stderr excerpt:")
        print("\n".join(generate.stderr.splitlines()[:16]))
    if not generation_completed:
        raise RuntimeError(
            "Multicoordinate generator failed; see stderr excerpt above."
        )

Temporary workspace: temporary directory
Project directory exists before generation: False


Generator return code: 0
Setting up reference_metric[Spherical_rfm_precompute]...
Setting up reference_metric[SinhCartesian_rfm_precompute]...
Setting up reference_metric[Cartesian_rfm_precompute]...
Setting up reference_metric[SinhSpherical_rfm_precompute]...
Evolved gridfunction "uu" has parity type 0.
Evolved gridfunction "vv" has parity type 0.
Outputting non-core modules key = nrpy.infrastructures.BHaH.rfm_wrapper_functions to BHaH_defines.h
Finished! Now go into project/wave_equation_multicoordinates and type `make` to build, then ./wave_equation_multicoordinates to run.
    Parameter file can be found in wave_equation_multicoordinates.par


## Generated File Inventory

The generated project should contain a Makefile, a default parameter file, C-family source files, and generated headers. Coordinate-system evidence is summarized from the generator constants and generated files without printing absolute paths.

In [4]:
if not generation_completed:
    print("Generated project inventory is unavailable because generation did not complete.")
elif not project_path.exists():
    print("Generated project directory is not present.")
else:
    required_paths = [
        project_path / "Makefile",
        project_path / "wave_equation_multicoordinates.par",
    ]
    for path in required_paths:
        print(path.relative_to(project_path), "exists:", path.exists())

    source_suffixes = {".c", ".cpp", ".cu"}
    header_suffixes = {".h", ".hpp", ".cuh"}
    generated_files = sorted(path for path in project_path.rglob("*") if path.is_file())
    source_files = [path for path in generated_files if path.suffix in source_suffixes]
    header_files = [path for path in generated_files if path.suffix in header_suffixes]

    print("Generated file count:", len(generated_files))
    print("C-family source count:", len(source_files))
    print("Header count:", len(header_files))
    print("First generated files:")
    for path in generated_files[:12]:
        print(" ", path.relative_to(project_path))

    coordinate_systems = sorted(generator_config.get("set_of_CoordSystems", []))
    print("Coordinate systems from generator constants:", coordinate_systems)
    coordinate_hits = {}
    for coord_system in coordinate_systems:
        hits = []
        for path in generated_files:
            relpath = path.relative_to(project_path)
            if coord_system in relpath.parts or f"__rfm__{coord_system}" in path.name:
                hits.append(str(relpath))
            if len(hits) >= 4:
                break
        coordinate_hits[coord_system] = hits

    print("Coordinate evidence:")
    for coord_system, hits in coordinate_hits.items():
        print(" ", coord_system + ":", hits)

Makefile exists: True
wave_equation_multicoordinates.par exists: True
Generated file count: 91
C-family source count: 81
Header count: 8
First generated files:
  BHaH_defines.h
  BHaH_function_prototypes.h
  Cartesian/apply_bcs_outerradiation_and_inner__rfm__Cartesian.c
  Cartesian/bcstruct_set_up__rfm__Cartesian.c
  Cartesian/ds_min_single_pt__rfm__Cartesian.c
  Cartesian/numerical_grid_params_Nxx_dxx_xx__rfm__Cartesian.c
  Cartesian/rfm_precompute_defines__rfm__Cartesian.c
  Cartesian/rfm_precompute_free__rfm__Cartesian.c
  Cartesian/rfm_precompute_malloc__rfm__Cartesian.c
  Cartesian/rhs_eval__rfm__Cartesian.c
  Cartesian/xx_to_Cart__rfm__Cartesian.c
  Makefile
Coordinate systems from generator constants: ['Cartesian', 'SinhCartesian', 'SinhSpherical', 'Spherical']
Coordinate evidence:
  Cartesian: ['Cartesian/apply_bcs_outerradiation_and_inner__rfm__Cartesian.c', 'Cartesian/bcstruct_set_up__rfm__Cartesian.c', 'Cartesian/ds_min_single_pt__rfm__Cartesian.c', 'Cartesian/numerical_grid

## Build and Run

The generated project is built with `make`. For a compact notebook run, the parameter file is adjusted to stop at a short final time before the executable is launched. This changes runtime parameters only; generated C files are left untouched.

In [5]:
import re
import shutil


def clean_process_output(text, max_lines=24):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    text = text.replace("\r", "\n")
    text = text.replace(str(project_path), "project/wave_equation_multicoordinates")
    text = text.replace(str(workspace_path), "temporary workspace")
    lines = [line.rstrip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines[:max_lines])


if not generation_completed or not project_path.exists():
    print("Build and run skipped because project generation did not complete.")
elif shutil.which("make") is None:
    print("Build and run skipped because make is not available in this environment.")
else:
    make_result = subprocess.run(
        ["make", "-j2"],
        cwd=project_path,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=180,
        check=False,
    )
    print("make return code:", make_result.returncode)
    print(clean_process_output(make_result.stdout, max_lines=18))

    executable = project_path / "wave_equation_multicoordinates"
    parfile = project_path / "wave_equation_multicoordinates.par"
    if make_result.returncode != 0:
        print("Executable run skipped because the generated project did not build.")
    elif not executable.exists():
        print("Executable run skipped because the expected executable was not produced.")
    else:
        par_text = parfile.read_text()
        par_text = re.sub(r"^t_final\s*=.*$", "t_final = 0.05", par_text, flags=re.MULTILINE)
        par_text = re.sub(
            r"^diagnostics_output_every\s*=.*$",
            "diagnostics_output_every = 0.01",
            par_text,
            flags=re.MULTILINE,
        )
        parfile.write_text(par_text)
        run_result = subprocess.run(
            [str(executable), str(parfile)],
            cwd=project_path,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=120,
            check=False,
        )
        print("run return code:", run_result.returncode)
        print(clean_process_output(run_result.stdout, max_lines=24))
        diagnostics = sorted(project_path.glob("out0d-grid*.txt"))
        print("0D diagnostic files:", [path.name for path in diagnostics[:4]])
        if diagnostics:
            sample = diagnostics[0].read_text().splitlines()[:6]
            print("sample diagnostic rows:")
            print("\n".join(sample))

make return code: 0
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs_inner_only.c -o apply_bcs_inner_only.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs_inner_only_specific_gfs.c -o apply_bcs_inner_only_specific_gfs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs_outerextrap_and_inner.c -o apply_bcs_outerextrap_and_inner.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs_outerradiation_and_inner.c -o apply_bcs_outerradiation_and_inner.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c bcstruct_set_up.c -o bcstruct_set_up.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c Cartesian/apply_bcs_outerradiation_and_inner__rfm__Cartesian.c -o Cartesian/apply_bcs_outerradiation_and_inner__rfm__Cartesian.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c Cartesian/bcstruct_set_up__rfm__Cartesian.c -o Cartesian/bcstruct_set_up__rfm__Cartesian.o
cc -std=gnu99 -O2 -march=native -g -Wall -I

run return code: 0
It: 0 t=0.000 / 0.1 = 0.00% dt=1/241.8 | t/h=0.00 ETA 0h00m00s
WRITING CHECKPOINT: cd struct size = 168 time=0.000000e+00
FINISHED WRITING CHECKPOINT
It: 1 t=0.004 / 0.1 = 8.27% dt=1/241.8 | t/h=372.59 ETA 0h00m00s
It: 2 t=0.008 / 0.1 = 16.54% dt=1/241.8 | t/h=376.52 ETA 0h00m00s
It: 3 t=0.012 / 0.1 = 24.81% dt=1/241.8 | t/h=493.54 ETA 0h00m00s
It: 4 t=0.017 / 0.1 = 33.08% dt=1/241.8 | t/h=592.41 ETA 0h00m00s
It: 5 t=0.021 / 0.1 = 41.36% dt=1/241.8 | t/h=521.17 ETA 0h00m00s
It: 6 t=0.025 / 0.1 = 49.63% dt=1/241.8 | t/h=582.15 ETA 0h00m00s
It: 7 t=0.029 / 0.1 = 57.90% dt=1/241.8 | t/h=543.12 ETA 0h00m00s
It: 8 t=0.033 / 0.1 = 66.17% dt=1/241.8 | t/h=585.91 ETA 0h00m00s
It: 9 t=0.037 / 0.1 = 74.44% dt=1/241.8 | t/h=629.25 ETA 0h00m00s
It: 10 t=0.041 / 0.1 = 82.71% dt=1/241.8 | t/h=586.69 ETA 0h00m00s
It: 11 t=0.045 / 0.1 = 90.98% dt=1/241.8 | t/h=619.37 ETA 0h00m00s
It: 12 t=0.050 / 0.1 = 99.25% dt=1/241.8 | t/h=587.17 ETA 0h00m00s
0D diagnostic files: ['out0d-grid00-C

## Related Notebooks

[Curvilinear wave equation](wave_equation_curvilinear.ipynb) introduces the single-coordinate curvilinear generator. [Curvilinear boundary conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb) explains the ghost-zone and parity calls used here.

## Where This Leads

- [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb) reviews the prerequisite step.
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.